In [14]:
#importing module
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import statsmodels.api as sm
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [4]:
pip install imbalanced-learn 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
#data load
df = pd.read_csv('tested.csv')
#show data 
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [9]:
# Defining feature
X = df.drop('Survived', axis=1)
y = df['Survived']

In [10]:
# Identify numeric categorical columns
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

In [15]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

In [16]:
# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
#used preprocessing
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

In [24]:
# X_train_prepared is a NumPy array; convert to a DataFrame (or use pd.isnull) to count missing values
print(pd.DataFrame(X_train_prepared).isnull().sum())


0     0
1    69
2     0
3     0
4     1
5     0
6     0
7     0
dtype: int64


In [25]:
from sklearn.impute import SimpleImputer

# Numeric columns: fill NaN with median
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_prepared)


In [26]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imputed, y_train)


In [30]:
# -----------------------------
# Logistic Regression
# -----------------------------
logit_clf = LogisticRegression(max_iter=1000)
logit_clf.fit(X_train_smote, y_train_smote)
y_pred = logit_clf.predict(X_test_imputed)

print("Logistic Regression Performance (SMOTE):")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


Logistic Regression Performance (SMOTE):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        53
           1       1.00      1.00      1.00        31

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84

[[53  0]
 [ 0 31]]


In [28]:
# Impute test set using the same imputer as training
X_test_imputed = imputer.transform(X_test_prepared)

# Predict
y_pred = logit_clf.predict(X_test_imputed)


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_prepared)
X_test_imputed = imputer.transform(X_test_prepared)


In [ ]:
if 'X_train_imputed' in globals() and 'X_test_imputed' in globals():
	X_train_encoded = pd.DataFrame(X_train_imputed, columns=cols)
	X_test_encoded = pd.DataFrame(X_test_imputed, columns=cols)
else:
	X_train_encoded = pd.DataFrame(X_train_prepared, columns=cols)
	X_test_encoded = pd.DataFrame(X_test_prepared, columns=cols)
